# AeroTwin Virtual Engine Integration

This notebook integrates Ninaad's `virtual-engine` simulation data (which has been validated against NASA C-MAPSS) directly into our AI/ML pipeline for RUL Estimation, Anomaly Detection, and SHAP Explainability.

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from sklearn.preprocessing import MinMaxScaler
from sklearn.ensemble import IsolationForest
import xgboost as xgb

# Note: In Colab, you would upload the 'dataset/all_missions.csv' generated by Ninaad's script.
# For this notebook, we assume it's available in the local path.
DATA_PATH = '../../dataset/all_missions.csv'
df = pd.read_csv(DATA_PATH)
print(f"Loaded {len(df)} telemetry records across {df['mission_id'].nunique()} missions.")
df.head()

## 1. RUL Label Generation & Preprocessing

In [ ]:
def calculate_rul(group):
    # Assuming the end of the mission is failure/end-of-life.
    max_tick = group['tick'].max()
    group['rul'] = max_tick - group['tick']
    return group

df = df.groupby('mission_id').apply(calculate_rul).reset_index(drop=True)

# Features from Ninaad's virtual engine
features = ['rpm', 'cht_celsius', 'egt_celsius', 'oil_pressure_bar', 'vibration', 'fuel_flow']

scaler = MinMaxScaler()
df[features] = scaler.fit_transform(df[features])

# Train/Test split by mission_id (e.g. healthy_0 for test, others for train)
train_df = df[df['mission_id'] != 'healthy_0']
test_df = df[df['mission_id'] == 'healthy_0']

X_train, y_train = train_df[features], train_df['rul']
X_test, y_test = test_df[features], test_df['rul']

print(f"Training set shape: {X_train.shape}")

## 2. Model Training (Random Forest / XGBoost RUL)

In [ ]:
model = xgb.XGBRegressor(n_estimators=100, max_depth=5, learning_rate=0.1, random_state=42)
model.fit(X_train, y_train)

preds = model.predict(X_test)

plt.figure(figsize=(10, 5))
plt.plot(y_test.values, label='Actual RUL', color='cyan')
plt.plot(preds, label='Predicted RUL (XGBoost)', color='magenta', linestyle='dashed')
plt.title("RUL Prediction on Ninaad's C-MAPSS Validated Engine Data")
plt.legend()
plt.grid(alpha=0.3)
plt.show()

## 3. Unsupervised Anomaly Detection (Isolation Forest)

In [ ]:
# Filter for only healthy flights for training
healthy_data = df[df['label_fault_type'] == 'healthy'][features].values

iso_forest = IsolationForest(contamination=0.01, random_state=42)
iso_forest.fit(healthy_data)

# Predict on a faulty mission
faulty_mission = df[df['mission_id'] == 'fault_oil_pressure_drop_0'].copy()
faulty_mission['anomaly_score'] = iso_forest.decision_function(faulty_mission[features].values)

plt.figure(figsize=(10, 3))
plt.plot(faulty_mission['tick'], faulty_mission['anomaly_score'], color='orange')
plt.axhline(0, color='red', linestyle='--', label='Anomaly Threshold')
plt.title("Anomaly Score during Oil Pressure Drop Fault")
plt.xlabel("Tick")
plt.legend()
plt.grid(alpha=0.3)
plt.show()